# HSK-5 Tutor — QLoRA training (Colab)

Fine-tunes **Qwen2.5-7B-Instruct** into the tutor with a LoRA adapter, then lets you download it.

**Before running:** set the runtime to a GPU — *Runtime → Change runtime type → GPU* (L4 or A100 recommended for a 7B; a free T4 works but is slower).

You'll upload four files from the repo: `config.py`, `train.py`, and the generated `data/train.jsonl` + `data/eval.jsonl`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install training deps
(Colab already ships torch; this tops up the rest.)

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" \
    "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"

## 3. Upload the project files
Run the cell, then pick **`config.py`**, **`train.py`**, **`train.jsonl`**, and **`eval.jsonl`** from your machine. The data files are moved into `data/` so `config.py`'s paths resolve.

In [ ]:
import os, shutil
from google.colab import files

uploaded = files.upload()  # choose config.py, train.py, train.jsonl, eval.jsonl

os.makedirs('data', exist_ok=True)
for name in ('train.jsonl', 'eval.jsonl'):
    if name in uploaded:
        shutil.move(name, f'data/{name}')

assert os.path.exists('config.py') and os.path.exists('train.py'), 'need config.py + train.py'
assert os.path.exists('data/train.jsonl'), 'need data/train.jsonl'
print('files ready:', sorted(os.listdir('.')), '| data:', sorted(os.listdir('data')))

## 4. Sanity run (optional but recommended)
A ~20-step run to confirm everything loads and the loss moves, before the full train. Takes a couple minutes.

In [ ]:
!python train.py --max-steps 20

## 5. Full training run
~30–60 min on an L4/A100 for ~800 examples × 3 epochs.

In [ ]:
!python train.py

## 6. Download the adapter
Zips `outputs/` (the LoRA adapter + tokenizer — only a few MB) and downloads it. Back on your Mac, unzip into the repo's `outputs/` for the merge + demo steps.

In [ ]:
shutil.make_archive('hsk5_adapter', 'zip', 'outputs')
files.download('hsk5_adapter.zip')